# Brute Force Approach for Minimum fuel Trajectories in Earth Moon System

This notebook applies a brute force approach to solve the problem of launching a rocket from Low Earth Orbit (LEO) to Low Moon Orbit (LMO).

Two impulsive burns are appied. One at LEO and one at LMO.

The time of flight and phase of departure are also optimized

### Imports

In [1]:
import numpy as np
import pandas as pd
from cr3bp import (
    create_earth_moon_system,
    grid_search_method
)

### Initialize the System / Problem

In [2]:
# Create the Earth-Moon system using the cr3bp module
em = create_earth_moon_system()
print(em.info())

CR3BP System Information:
  Primary 1 mass: 5.972e+24 kg
  Primary 2 mass: 7.342e+22 kg
  Primary 1 radius: 6.371e+06 m
  Primary 2 radius: 1.737e+06 m
  Total mass: 6.045e+24 kg
  Distance: 3.844e+08 m (384400.0 km)
  Mass parameter μ: 0.012145

Characteristic scales:
  Length (l*): 3.844e+08 m (384400.0 km)
  Time (t*): 3.752e+05 s (4.343 days)
  Velocity (v*): 1.025e+03 m/s (1.025 km/s)
  Acceleration (a*): 2.731e-03 m/s^2
  Period: 27.285 days
None


In [3]:
# Define the LEO and LMO altitudes in meters
leo_alt_m=463e3
lmo_alt_m=100e3

## Run the Optimization Method

In [14]:
optimals = np.load("grid_search_results_constraint_optimal.npy")
optimal_theta = optimals[0]
optimal_delta_v = optimals[1]
optimal_alpha = optimals[2]
optimal_tof = optimals[3]

In [15]:
print(optimals)

[3.90820513 3.03578947 0.08684211 0.65615385]


In [ ]:
#dec_var_ranges = 
#dec_var_ranges = new_ranges
dec_var_ranges = [[optimal_theta-0.05, optimal_theta+0.05], [optimal_delta_v-0.05, optimal_delta_v+0.05], [optimal_alpha-0.05, optimal_alpha+0.05], [optimal_tof-0.05, optimal_tof+0.05]]

In [ ]:
results_df = grid_search_method(em, dec_var_ranges, 2.6e-3, leo_alt_m, lmo_alt_m)

Performing grid search with 1 x 1 x 1 x 1600 = 1600 grid points...


Iteration: 790/1600
Grid point satisfies constraint: Theta=3.94 rad, Delta_v=3.03, Delta_v_angle=0.07 rad, TOF=0.68 s => Distance to LMO=994.67 km, Total Delta_v=3.71 km/s
New optimal found: Delta_v=3.10 km/s, Theta=3.94 rad, Delta_v_angle=0.07 rad, TOF=0.68 s, Distance to LMO=994.67 km
Iteration: 791/1600
Grid point satisfies constraint: Theta=3.94 rad, Delta_v=3.03, Delta_v_angle=0.07 rad, TOF=0.68 s => Distance to LMO=974.78 km, Total Delta_v=3.67 km/s
New optimal found: Delta_v=3.10 km/s, Theta=3.94 rad, Delta_v_angle=0.07 rad, TOF=0.68 s, Distance to LMO=974.78 km
Iteration: 792/1600
Grid point satisfies constraint: Theta=3.94 rad, Delta_v=3.03, Delta_v_angle=0.07 rad, TOF=0.68 s => Distance to LMO=957.48 km, Total Delta_v=3.62 km/s
New optimal found: Delta_v=3.10 km/s, Theta=3.94 rad, Delta_v_angle=0.07 rad, TOF=0.68 s, Distance to LMO=957.48 km
Iteration: 793/1600
Grid point satisfies constraint: Theta=3.94 rad, Delta_v=3.03, Delta_v_angle=0.07 rad, TOF=0.68 s => Distance to LMO

In [8]:
print(results_df)

       theta   delta_v  delta_v_angle       tof  total_delta_v  \
0   3.941538  3.030526       0.065789  0.677598       3.175301   
1   3.941538  3.030526       0.065789  0.677661       3.180252   
2   3.941538  3.030526       0.065789  0.677536       3.194993   
3   3.941538  3.030526       0.065789  0.677724       3.206375   
4   3.941538  3.030526       0.065789  0.677473       3.232705   
5   3.941538  3.030526       0.065789  0.677786       3.244367   
6   3.941538  3.030526       0.065789  0.677411       3.280081   
7   3.941538  3.030526       0.065789  0.677849       3.287279   
8   3.941538  3.030526       0.065789  0.677348       3.331743   
9   3.941538  3.030526       0.065789  0.677286       3.384633   
10  3.941538  3.030526       0.065789  0.677223       3.436914   
11  3.941538  3.030526       0.065789  0.677161       3.487400   
12  3.941538  3.030526       0.065789  0.677098       3.535300   
13  3.941538  3.030526       0.065789  0.677036       3.580089   
14  3.9415

In [9]:
optimals = results_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values

In [ ]:
np.save("grid_search_results_new.npy", optimals)
np.save("grid_search_results_df_new.npy", results_df)

In [11]:
shrink = 0.5
new_ranges = []
for i in range(4):
    current_span = dec_var_ranges[i][1] - dec_var_ranges[i][0]
    half_width = current_span * shrink / 2
    new_min = max(dec_var_ranges[i][0], optimals[i] - half_width)
    new_max = min(dec_var_ranges[i][1], optimals[i] + half_width)
    new_ranges.append([new_min, new_max])
print(new_ranges)

[[np.float64(3.9415384615384617), np.float64(3.9415384615384617)], [np.float64(3.0305263157894737), np.float64(3.0305263157894737)], [np.float64(0.06578947368421054), np.float64(0.06578947368421054)], [np.float64(0.6525984990619137), np.float64(0.7025984990619137)]]


In [12]:
results_df['distance_to_lmo'] = results_df['distance_to_lmo'] * em.l_star

In [13]:
print(results_df)

       theta   delta_v  delta_v_angle       tof  total_delta_v  \
0   3.941538  3.030526       0.065789  0.677598       3.175301   
1   3.941538  3.030526       0.065789  0.677661       3.180252   
2   3.941538  3.030526       0.065789  0.677536       3.194993   
3   3.941538  3.030526       0.065789  0.677724       3.206375   
4   3.941538  3.030526       0.065789  0.677473       3.232705   
5   3.941538  3.030526       0.065789  0.677786       3.244367   
6   3.941538  3.030526       0.065789  0.677411       3.280081   
7   3.941538  3.030526       0.065789  0.677849       3.287279   
8   3.941538  3.030526       0.065789  0.677348       3.331743   
9   3.941538  3.030526       0.065789  0.677286       3.384633   
10  3.941538  3.030526       0.065789  0.677223       3.436914   
11  3.941538  3.030526       0.065789  0.677161       3.487400   
12  3.941538  3.030526       0.065789  0.677098       3.535300   
13  3.941538  3.030526       0.065789  0.677036       3.580089   
14  3.9415